# TDM Calculator — Toy Example

A sandbox implementation of five TDM strategies against a dummy TAZ dataset. Formulas and elasticities come from `Methods_Research_Updated.xlsx`.

**Strategies included**
1. Establishing a new TMO (CAPCOA voluntary CTR approach)
2. Parking Fees / Curb Management (with trip-purpose selector)
3. Increased Frequency — Local Transit
4. Flexible Schedules / Telework Policy
5. Vanpool (employer-based)

**Design notes**
- Each strategy is implemented as a pure function that accepts a DataFrame of one or more TAZs plus user inputs and returns a per-TAZ VMT-reduction table.
- Strategies are applied *independently* here. A placeholder stacking section at the bottom flags the additive-vs-multiplicative question for later.
- All effect sizes are toy values; replace with calibrated Colorado-specific figures before production use.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

## 1. Dummy TAZ dataset

Ten TAZs with a mix of urban / suburban / employment-center / residential profiles. Columns:

| Column | Meaning |
|---|---|
| `taz_id` | TAZ identifier |
| `area_type` | Urban / Suburban / Rural — narrative only |
| `population`, `households`, `employment` | Stock variables |
| `daily_vmt` | Total daily VMT produced/attracted in the TAZ |
| `commute_vmt` | Daily VMT attributable to commute trips |
| `recreational_vmt` | Daily VMT attributable to recreational / non-work trips |
| `other_vmt` | Remainder (shopping, school, etc.) — derived |
| `transit_mode_share`, `vehicle_mode_share` | Share of person-trips |
| `avo` | Average vehicle occupancy |
| `avg_trip_length`, `avg_commute_length` | Miles, one-way |
| `share_emp_paying_parking` | Share of employees currently paying to park |
| `current_parking_price` | $ per day (typical) |
| `transit_vrh` | Daily transit vehicle revenue hours serving the TAZ |
| `transit_vrm` | Daily transit vehicle revenue miles serving the TAZ |

In [ ]:
taz = pd.DataFrame([
    # taz, type,        pop,   hh,    emp,    daily_vmt, commute_vmt, rec_vmt, transit_ms, veh_ms, avo,  trip_len, commute_len, share_pay, price, vrh,  vrm
    ('T01', 'Urban-Core',   8500,  4100,  12000,  52000,  18500,  9500,  0.22, 0.62, 1.15,  3.2, 4.8, 0.75, 12.0, 180, 2400),
    ('T02', 'Urban-Core',   6200,  3000,   9500,  41000,  15000,  7200,  0.19, 0.66, 1.18,  3.4, 5.2, 0.60,  9.0, 140, 1900),
    ('T03', 'Urban-Mixed',  9800,  4300,   5400,  46000,  14000,  9800,  0.12, 0.75, 1.22,  4.1, 6.0, 0.30,  5.0,  90, 1300),
    ('T04', 'Suburban-Emp', 3200,  1200,  15000,  68000,  28000,  8500,  0.08, 0.82, 1.20,  6.8, 9.5, 0.20,  3.5,  55,  950),
    ('T05', 'Suburban-Res',11500,  4600,   2100,  55000,  13500, 14500,  0.06, 0.86, 1.28,  7.5, 11.0,0.05,  0.0,  40,  720),
    ('T06', 'Suburban-Res', 9200,  3700,   1800,  47000,  11000, 12500,  0.05, 0.87, 1.30,  7.8, 11.5,0.05,  0.0,  30,  560),
    ('T07', 'Suburban-Mix', 7400,  3000,   6200,  51000,  16500, 10200,  0.09, 0.80, 1.24,  5.9, 8.2, 0.25,  4.0,  70, 1100),
    ('T08', 'Employment',   1400,   500,  22000,  84000,  38000,  6000,  0.15, 0.75, 1.17,  6.2, 9.0, 0.55, 11.0, 110, 1700),
    ('T09', 'Rural',        2800,  1100,    900,  28000,   6500,  7800,  0.02, 0.92, 1.32, 10.5, 14.0,0.05,  0.0,  10,  220),
    ('T10', 'Mixed-Use',    5600,  2400,   7800,  44000,  14000,  9200,  0.14, 0.71, 1.20,  4.6, 6.8, 0.40,  6.5,  95, 1450),
], columns=[
    'taz_id','area_type','population','households','employment',
    'daily_vmt','commute_vmt','recreational_vmt',
    'transit_mode_share','vehicle_mode_share','avo',
    'avg_trip_length','avg_commute_length',
    'share_emp_paying_parking','current_parking_price',
    'transit_vrh','transit_vrm'
])
taz['other_vmt'] = taz['daily_vmt'] - taz['commute_vmt'] - taz['recreational_vmt']
taz

In [ ]:
def select_tazs(df, taz_ids):
    """Return rows for one or more TAZ ids. Accepts a single id string or a list."""
    if isinstance(taz_ids, str):
        taz_ids = [taz_ids]
    out = df[df['taz_id'].isin(taz_ids)].copy()
    missing = set(taz_ids) - set(out['taz_id'])
    if missing:
        raise ValueError(f'Unknown taz_id(s): {sorted(missing)}')
    return out.reset_index(drop=True)

## Strategy 1 — Establishing a new TMO

**Approach:** CAPCOA Voluntary Commute Trip Reduction Program (methods sheet row 20).

**Formula**
$$\%\,\Delta\text{Commute VMT} = -\bigl(\text{share}_{\text{after}} - \text{share}_{\text{before}}\bigr) \times r_{\text{CTR}}$$

- $r_{\text{CTR}}$ = 4% commute-VMT reduction per eligible employee (CAPCOA TRT-1 voluntary CTR midpoint).
- Inputs: share of population/employees in the TMO **before** and **after**.
- Applied to commute VMT only.

In [ ]:
CTR_EFFECT = 0.04  # CAPCOA voluntary CTR midpoint

def strategy_tmo(tazs, share_before, share_after, ctr_effect=CTR_EFFECT):
    """
    share_before, share_after : float in [0,1], share of employees covered by TMO programs.
    Returns a dataframe keyed by taz_id.
    """
    delta_share = share_after - share_before
    pct_reduction = delta_share * ctr_effect  # fraction of commute VMT removed
    out = tazs[['taz_id','commute_vmt']].copy()
    out['strategy'] = 'TMO (new)'
    out['inputs'] = f'share {share_before:.0%}→{share_after:.0%}, r_CTR={ctr_effect:.0%}'
    out['pct_commute_vmt_reduction'] = pct_reduction
    out['commute_vmt_reduction'] = out['commute_vmt'] * pct_reduction
    out['daily_vmt_reduction'] = out['commute_vmt_reduction']
    return out

In [ ]:
# Demo: establish a TMO in two employment-heavy TAZs, growing coverage from 0% to 40%
strategy_tmo(select_tazs(taz, ['T04','T08']), share_before=0.0, share_after=0.40)

## Strategy 2 — Parking Fees / Curb Management

**Approach:** methods sheet row 16. Price elasticity of parking demand is -0.4 (Pierce & Shoup 2013 / SFPark).

**Formula**
$$\%\,\Delta\text{VMT}_{\text{purpose}} = \varepsilon_p \times \%\,\Delta\text{Price} \times \text{share}_{\text{trips affected}} \times \text{VMT-to-trip ratio}$$

The `trip_purpose` selector restricts the VMT base the elasticity is applied to:
- `'all'` → daily_vmt
- `'workplace'` → commute_vmt
- `'recreational'` → recreational_vmt

Inputs: existing price, new price, share of trips whose parking price actually changes (e.g. share of on-street curb trips under the new meter zone).

> **Caveats.**
> - The linear elasticity approximation is only reliable for modest price changes (roughly ±50%). For large hikes (e.g. introducing $18 pricing where parking was $3.50, a ~400% jump) the formula will overstate reduction — use a log/arc elasticity or a saturation cap before production.
> - When `current_parking_price` is **$0** (free parking), the calculator imputes a baseline of `new_price / 2` so introducing pricing is modeled as a 100% price increase (rather than producing NaN). Override `existing_price=` explicitly if you have a better baseline.

In [ ]:
PARKING_ELASTICITY = -0.4  # Pierce & Shoup 2013

_VMT_COL = {'all': 'daily_vmt', 'workplace': 'commute_vmt', 'recreational': 'recreational_vmt'}

def strategy_parking_price(tazs, new_price, trip_purpose='all',
                           share_trips_affected=1.0,
                           vmt_to_trip_ratio=1.0,
                           elasticity=PARKING_ELASTICITY,
                           existing_price=None):
    """
    new_price            : $/day under the policy
    trip_purpose         : 'all' | 'workplace' | 'recreational'
    share_trips_affected : fraction of trips in that purpose that actually see the price change
    vmt_to_trip_ratio    : scaling between trip-share reduction and VMT reduction (1.0 ≈ neutral)
    existing_price       : if None, taken from each TAZ's current_parking_price column
    """
    if trip_purpose not in _VMT_COL:
        raise ValueError(f"trip_purpose must be one of {list(_VMT_COL)}")
    vmt_col = _VMT_COL[trip_purpose]
    out = tazs[['taz_id', vmt_col]].copy()
    old = tazs['current_parking_price'] if existing_price is None else pd.Series(existing_price, index=tazs.index)
    # $0 baseline → treat existing price as new_price/2 so the policy is
    # modeled as a 100% increase (rather than producing NaN). Override
    # `existing_price=` explicitly if you have a better free-parking baseline.
    effective_old = np.where(old > 0, old, new_price / 2.0)
    pct_price_change = (new_price - effective_old) / effective_old
    pct_reduction = elasticity * pct_price_change * share_trips_affected * vmt_to_trip_ratio
    out['strategy'] = f'Parking Price ({trip_purpose})'
    out['inputs'] = [
        (f'${o:.2f}→${new_price:.2f} (free→priced, modeled as ${new_price/2:.2f} baseline)' if o == 0
         else f'${o:.2f}→${new_price:.2f}')
        + f', affected={share_trips_affected:.0%}'
        for o in old
    ]
    out['pct_vmt_reduction'] = pct_reduction  # negative = reduction
    out['vmt_reduction'] = -out[vmt_col] * pct_reduction  # positive = miles saved
    out['daily_vmt_reduction'] = out['vmt_reduction']
    return out

In [ ]:
# Demo A: raise workplace parking prices by ~50% in downtown TAZs, affecting 80% of commute trips
workplace = strategy_parking_price(
    select_tazs(taz, ['T01','T02','T08']),
    new_price=18.0, trip_purpose='workplace', share_trips_affected=0.80,
)
workplace

In [ ]:
# Demo B: introduce $4/hr curb pricing in mixed-use TAZs, targeting recreational trips
rec = strategy_parking_price(
    select_tazs(taz, ['T01','T03','T10']),
    new_price=8.0, trip_purpose='recreational', share_trips_affected=0.50,
)
rec

## Strategy 3 — Increased Frequency (Local Transit)

**Approach:** methods sheet row 0. Handy et al. 2013 — elasticity of transit ridership w.r.t. service ≈ 0.5.

**Formula**
$$\%\,\Delta\text{VMT} = -L \times \frac{\%\,\Delta\text{Service} \times \text{transit MS} \times \varepsilon \times (1/\text{AVO})}{\text{vehicle MS}}$$

- $L$ = level of implementation (share of TAZ service affected), 0–1.
- Service change can be computed from VRH or VRM deltas (`service_basis='vrh'` or `'vrm'`) when passing absolute new service; or specified directly as a `pct_service_increase`.
- Applied to daily VMT (mode shift reduces total trips, not just commute).

In [ ]:
TRANSIT_ELASTICITY = 0.5  # Handy et al. 2013

def strategy_transit_frequency(tazs, level_of_implementation=1.0,
                               pct_service_increase=None,
                               new_service=None, service_basis='vrh',
                               elasticity=TRANSIT_ELASTICITY):
    """
    Provide EITHER pct_service_increase (fraction, e.g. 0.25 for +25%),
    OR new_service (absolute VRH/VRM) together with service_basis in {'vrh','vrm'}.
    """
    out = tazs[['taz_id','daily_vmt','transit_mode_share','vehicle_mode_share','avo']].copy()
    if pct_service_increase is None:
        if new_service is None:
            raise ValueError('Provide pct_service_increase or new_service')
        base_col = {'vrh':'transit_vrh','vrm':'transit_vrm'}[service_basis]
        pct = (new_service - tazs[base_col]) / tazs[base_col]
    else:
        pct = pd.Series(pct_service_increase, index=tazs.index)
    pct_reduction = -level_of_implementation * (
        pct * tazs['transit_mode_share'] * elasticity * (1.0 / tazs['avo'])
    ) / tazs['vehicle_mode_share']
    out['strategy'] = 'Transit Frequency'
    out['inputs'] = [f'L={level_of_implementation:.0%}, Δservice={p:.0%}, ε={elasticity}' for p in pct]
    out['pct_vmt_reduction'] = pct_reduction
    out['daily_vmt_reduction'] = -out['daily_vmt'] * pct_reduction
    return out[['taz_id','strategy','inputs','daily_vmt','pct_vmt_reduction','daily_vmt_reduction']]

In [ ]:
# Demo: 25% frequency bump on routes covering 60% of each selected TAZ
strategy_transit_frequency(
    select_tazs(taz, ['T01','T02','T07','T10']),
    level_of_implementation=0.60,
    pct_service_increase=0.25,
)

In [ ]:
# Alternative demo: expressed as a move from current VRH to a target VRH
sel = select_tazs(taz, ['T01','T02'])
strategy_transit_frequency(sel, level_of_implementation=1.0,
                           new_service=sel['transit_vrh'] * 1.25, service_basis='vrh')

## Strategy 4 — Flexible Schedules / Telework

**Approach:** methods sheet row 23.

The spreadsheet formula as written multiplies by average trip length, which yields miles rather than a percentage. Interpreted as the intent — telework eliminates the commute round trip on telework days — the reduction in commute VMT is:

$$\%\,\Delta\text{Commute VMT} = -\,\text{share}_{\text{eligible}} \times \frac{\text{telework days/wk}}{5}$$

Absolute daily VMT reduction can also be cross-checked against `eligible_employees × telework_days/5 × 2 × avg_commute_length`; both are exposed so the team can compare.

In [ ]:
def strategy_telework(tazs, share_eligible, telework_days_per_week):
    """
    share_eligible         : fraction of employees with a telework option, 0-1
    telework_days_per_week : average days teleworked per eligible employee, 0-5
    """
    if not 0 <= telework_days_per_week <= 5:
        raise ValueError('telework_days_per_week must be 0-5')
    pct_reduction = -share_eligible * (telework_days_per_week / 5.0)
    out = tazs[['taz_id','employment','commute_vmt','avg_commute_length']].copy()
    out['strategy'] = 'Telework'
    out['inputs'] = f'eligible={share_eligible:.0%}, days/wk={telework_days_per_week}'
    out['pct_commute_vmt_reduction'] = pct_reduction
    # Percentage-based reduction (primary)
    out['commute_vmt_reduction'] = -out['commute_vmt'] * pct_reduction
    # Trip-based cross-check: eligible × days/5 × 2 one-way legs × commute length
    out['commute_vmt_reduction_tripbased'] = (
        out['employment'] * share_eligible * (telework_days_per_week/5.0) * 2 * out['avg_commute_length']
    )
    out['daily_vmt_reduction'] = out['commute_vmt_reduction']
    return out

In [ ]:
# Demo: 50% of employees eligible, teleworking 2 days/week, across three employment-heavy TAZs
strategy_telework(select_tazs(taz, ['T04','T08','T10']), share_eligible=0.50, telework_days_per_week=2)

## Strategy 5 — Vanpool (employer-based)

**Approach:** methods sheet row 18.

For a VMT-only calculator (emissions factors dropped), a vanpool participant replaces `avg_commute_length` of SOV VMT with `avg_commute_length / vanpool_occupancy` of van VMT. Per-participant VMT reduction fraction is therefore $1 - 1/n_{\text{van}}$.

$$\%\,\Delta\text{Commute VMT} = -\,\text{participation rate} \times \Bigl(1 - \tfrac{1}{\text{vanpool occupancy}}\Bigr)$$

Typical vanpool occupancy is 6–8 riders per van.

In [ ]:
def strategy_vanpool(tazs, participation_rate, vanpool_occupancy=7):
    """
    participation_rate : share of eligible employees in a vanpool, 0-1
    vanpool_occupancy  : average riders per van (>=2)
    """
    if vanpool_occupancy < 2:
        raise ValueError('vanpool_occupancy must be >= 2')
    pct_reduction = -participation_rate * (1 - 1/vanpool_occupancy)
    out = tazs[['taz_id','commute_vmt','avg_commute_length','employment']].copy()
    out['strategy'] = 'Vanpool (employer)'
    out['inputs'] = f'participation={participation_rate:.0%}, occ={vanpool_occupancy}'
    out['pct_commute_vmt_reduction'] = pct_reduction
    out['commute_vmt_reduction'] = -out['commute_vmt'] * pct_reduction
    out['daily_vmt_reduction'] = out['commute_vmt_reduction']
    return out

In [ ]:
# Demo: 8% of employees participate, avg 7 riders per van, long-commute suburban TAZs
strategy_vanpool(select_tazs(taz, ['T04','T05','T09']), participation_rate=0.08, vanpool_occupancy=7)

## Summary — apply several strategies to one or more TAZs

Each call below returns a long-format row per (TAZ, strategy). The summary stacks them for side-by-side comparison.

> **Stacking (future work).** Reductions below are **computed independently** — do **not** sum them to estimate combined impact. For multiplicative stacking, combined retained VMT = ∏(1 − rᵢ); this is only valid when the measures act on disjoint behavioral channels. Some pairs must not be stacked at all (e.g., Workplace Parking Pricing + Parking Cash-Out target the same decision). Leaving the dispatcher structure here as a hook for that logic later.

In [ ]:
def summarize(*strategy_results):
    """Stack per-strategy result frames into one long table, keeping common columns."""
    common = ['taz_id','strategy','inputs','daily_vmt_reduction']
    parts = []
    for r in strategy_results:
        parts.append(r[common].copy())
    summary = pd.concat(parts, ignore_index=True)
    summary = summary.merge(taz[['taz_id','daily_vmt']], on='taz_id', how='left')
    summary['pct_of_daily_vmt'] = summary['daily_vmt_reduction'] / summary['daily_vmt']
    return summary

In [ ]:
selected = ['T01','T04','T08']
sel_df = select_tazs(taz, selected)

results = [
    strategy_tmo(sel_df, share_before=0.0, share_after=0.40),
    strategy_parking_price(sel_df, new_price=18.0, trip_purpose='workplace', share_trips_affected=0.80),
    strategy_transit_frequency(sel_df, level_of_implementation=0.60, pct_service_increase=0.25),
    strategy_telework(sel_df, share_eligible=0.50, telework_days_per_week=2),
    strategy_vanpool(sel_df, participation_rate=0.08, vanpool_occupancy=7),
]

summary = summarize(*results)
summary.sort_values(['taz_id','strategy']).reset_index(drop=True)

In [ ]:
# Pivoted view: TAZ × strategy VMT reductions
summary.pivot_table(
    index='taz_id',
    columns='strategy',
    values='daily_vmt_reduction',
    aggfunc='sum',
).round(1)

## 7. Connecting to live CDOT 2019 TDM data

The toy `taz` table above is hand-crafted. This section pulls the **published CDOT 2019 TDM TAZ** feature service and reshapes it into the same column schema, so the strategy functions can be applied to real zones without any change.

**Service:** `https://services3.arcgis.com/gjVvdAtTsjMYfRZ8/arcgis/rest/services/Development_CDOT_2019_TDM_TAZ/FeatureServer/0`

The AGOL item is shared publicly, so this section talks to the REST endpoint directly with `requests` — no login required.

### What the TDM provides vs. what the calculator needs

| Calculator column | TDM source | Notes |
|---|---|---|
| `taz_id` | `TAZ_new_ID` | cast to string |
| `area_type` | `AreaType` | |
| `population` | `TOT_2019_POP` | |
| `households` | `TOTAL_2019_HHS` | |
| `employment` | `Empl_2019` | |
| `daily_vmt` | `VMT` | |
| `commute_vmt` / `recreational_vmt` / `other_vmt` | `VMT × purpose share` | **the TDM does not split VMT by trip purpose**, so we apply an explicit share assumption (`VMT_PURPOSE_SHARE` below) |
| `avg_trip_length` | `VMT / rptTrips` | derived |
| `transit_mode_share`, `vehicle_mode_share`, `avo`, `avg_commute_length`, `share_emp_paying_parking`, `current_parking_price`, `transit_vrh`, `transit_vrm` | — not in the TDM | flat defaults in `BEHAVIORAL_DEFAULTS`; tune per analysis or override per-TAZ |

The trip-purpose share and behavioral defaults are intentionally exposed at the top of the section so they can be tuned (e.g., from regional NHTS splits, the model's HBW/HBO/NHB trip totals × avg lengths, or NTD service data).


In [ ]:
# ============================================================
# CONFIG — live TDM hook + behavioral assumptions
# ============================================================
import requests

TDM_TAZ_URL = (
    "https://services3.arcgis.com/gjVvdAtTsjMYfRZ8/arcgis/rest/services/"
    "Development_CDOT_2019_TDM_TAZ/FeatureServer/0"
)

# Share of daily VMT attributed to each trip purpose. Must sum to 1.0.
# Defaults are rough NHTS-style splits; replace with calibrated CDOT figures
# (e.g., from the model's HBW / HBO / NHB / recreational trip totals × avg lengths).
VMT_PURPOSE_SHARE = {
    "commute":      0.30,   # HBW
    "recreational": 0.20,   # NHB social/rec
    "other":        0.50,   # HBO + remaining NHB (shopping, school, errands, business)
}

# Behavioral / service columns the TDM TAZ does not carry. Flat starting points
# so the strategy functions don't error out — override per-TAZ
# (e.g. `real_taz.loc[mask, "transit_mode_share"] = 0.20`) when local data is available.
BEHAVIORAL_DEFAULTS = {
    "transit_mode_share":       0.05,   # statewide-ish; bump for urban/CBD TAZs
    "vehicle_mode_share":       0.85,
    "avo":                      1.20,
    "avg_commute_length":       11.0,   # miles, one-way
    "share_emp_paying_parking": 0.10,
    "current_parking_price":    0.0,
    "transit_vrh":              0,
    "transit_vrm":              0,
}

assert abs(sum(VMT_PURPOSE_SHARE.values()) - 1.0) < 1e-9, "VMT_PURPOSE_SHARE must sum to 1"
print("VMT purpose share  :", VMT_PURPOSE_SHARE)
print("Behavioral defaults:", BEHAVIORAL_DEFAULTS)


In [ ]:
def fetch_tdm_taz(layer_url=TDM_TAZ_URL, where="1=1",
                  page_size=2000, return_geometry=False):
    """Fetch all TAZ attributes from the published CDOT 2019 TDM service.

    The AGOL item is shared publicly, so this hits the REST /query endpoint
    anonymously. Pages with `resultOffset` until `exceededTransferLimit` is
    False. Geometry is omitted by default — the calculator only needs
    attributes — but pass return_geometry=True for maps.
    """
    query_url = layer_url.rstrip("/") + "/query"
    rows, offset = [], 0
    while True:
        params = {
            "where":             where,
            "outFields":         "*",
            "returnGeometry":    str(return_geometry).lower(),
            "f":                 "json",
            "resultOffset":      offset,
            "resultRecordCount": page_size,
        }
        r = requests.get(query_url, params=params, timeout=120)
        r.raise_for_status()
        js = r.json()
        if "error" in js:
            raise RuntimeError(f"ArcGIS error: {js['error']}")
        feats = js.get("features", [])
        rows.extend(f["attributes"] for f in feats)
        if not js.get("exceededTransferLimit") or not feats:
            break
        offset += len(feats)
        print(f"  fetched {len(rows):,} features so far ...")
    print(f"  done: {len(rows):,} features")
    return pd.DataFrame(rows)


In [ ]:
def tdm_to_calculator(raw_df, vmt_share=None, defaults=None, taz_id_col="TAZ_new_ID"):
    """Reshape raw TDM TAZ attributes into the column schema used by the strategy functions.

    - Splits `VMT` into commute / recreational / other using `vmt_share`
      (defaults to VMT_PURPOSE_SHARE).
    - Derives `avg_trip_length` from VMT / rptTrips (0 where rptTrips is 0).
    - Fills behavioral columns the TDM does not carry from `defaults`
      (defaults to BEHAVIORAL_DEFAULTS).
    """
    vmt_share = vmt_share or VMT_PURPOSE_SHARE
    defaults  = defaults  or BEHAVIORAL_DEFAULTS

    out = pd.DataFrame()
    out["taz_id"]      = raw_df[taz_id_col].astype("Int64").astype(str)
    out["area_type"]   = raw_df["AreaType"].fillna("Unknown")
    out["population"]  = raw_df["TOT_2019_POP"].fillna(0)
    out["households"]  = raw_df["TOTAL_2019_HHS"].fillna(0)
    out["employment"]  = raw_df["Empl_2019"].fillna(0)
    out["daily_vmt"]   = raw_df["VMT"].fillna(0)

    out["commute_vmt"]      = out["daily_vmt"] * vmt_share["commute"]
    out["recreational_vmt"] = out["daily_vmt"] * vmt_share["recreational"]
    out["other_vmt"]        = out["daily_vmt"] * vmt_share["other"]

    trips = raw_df.get("rptTrips", pd.Series(np.nan, index=raw_df.index)).replace(0, np.nan)
    out["avg_trip_length"] = (out["daily_vmt"] / trips).fillna(0)

    for k, v in defaults.items():
        out[k] = v

    return out


In [ ]:
print("Pulling TAZ attributes from CDOT 2019 TDM service ...")
raw_taz  = fetch_tdm_taz()
real_taz = tdm_to_calculator(raw_taz)

print(f"\nLoaded {len(real_taz):,} TAZs.")
print(f"Statewide VMT (model 2019)               : {real_taz['daily_vmt'].sum():,.0f}")
print(f"Statewide commute VMT @ {VMT_PURPOSE_SHARE['commute']:.0%} share : {real_taz['commute_vmt'].sum():,.0f}")

print("\nTop 10 TAZs by daily VMT:")
real_taz.nlargest(10, "daily_vmt")[
    ["taz_id","area_type","population","employment","daily_vmt","commute_vmt","avg_trip_length"]
]


### Demo — apply the existing strategy functions to real CDOT TAZs

Pick the top-employment TAZs as a stand-in target for an employer-focused TDM package: TMO + workplace parking pricing + telework. The strategy functions are unchanged — they just consume real `commute_vmt` (= 30% of model VMT under the assumption above) instead of the toy values.

The parking-price demo overrides `current_parking_price` to a typical downtown figure ($10/day) for the selected TAZs, since the TDM does not provide parking prices.


In [ ]:
# Top 5 employment TAZs as a worked example
top_emp_ids = real_taz.nlargest(5, "employment")["taz_id"].tolist()
print("Top-employment TAZs:", top_emp_ids)

real_sel = select_tazs(real_taz, top_emp_ids)

# For the parking demo, set a non-zero baseline price (the default is $0).
real_sel_parking = real_sel.assign(current_parking_price=10.0)

results_real = [
    strategy_tmo(real_sel,
                 share_before=0.0, share_after=0.40),
    strategy_parking_price(real_sel_parking,
                           new_price=15.0, trip_purpose='workplace',
                           share_trips_affected=0.80),
    strategy_telework(real_sel,
                      share_eligible=0.50, telework_days_per_week=2),
]

# Inline summary — the toy `summarize` reaches the global `taz`; here we use the real frame.
common = ['taz_id','strategy','inputs','daily_vmt_reduction']
summary_real = pd.concat([r[common] for r in results_real], ignore_index=True)
summary_real = summary_real.merge(
    real_taz[['taz_id','area_type','employment','daily_vmt']],
    on='taz_id', how='left',
)
summary_real['pct_of_daily_vmt'] = summary_real['daily_vmt_reduction'] / summary_real['daily_vmt']
summary_real.sort_values(['taz_id','strategy']).reset_index(drop=True)
